# InternVL2 — Multimodal chat

InternVL2 is OpenGVLab's vision-language family with strong OCR and document understanding. This notebook uses the **2 B** variant (smallest), which fits on every AurumOS profile (~5 GB VRAM at bf16, ~1.5 GB at int4). For larger variants bump `MODEL_ID` to `InternVL2-8B` or `InternVL2-26B` if your profile permits.

InternVL2 has a custom forward pass (not directly LlavaForConditionalGeneration); we use `trust_remote_code=True` so HuggingFace pulls the model's own modeling files.

In [ ]:
import os

# Pick a model size that matches the user's profile. InternVL2's repo names
# are simple (-2B, -8B, -26B). Default to 2B so the notebook runs everywhere.
PROFILE = os.environ.get('AURUM_PROFILE', 'standard')
MODEL_ID = {
    'lite':        'OpenGVLab/InternVL2-1B',
    'standard':    'OpenGVLab/InternVL2-2B',
    'pro':         'OpenGVLab/InternVL2-8B',
    'workstation': 'OpenGVLab/InternVL2-26B',
}.get(PROFILE, 'OpenGVLab/InternVL2-2B')
print('Using:', MODEL_ID, '(profile:', PROFILE + ')')

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

# trust_remote_code is required: InternVL2 ships its own modeling_internvl_chat.py
# in the repo. The code is read-only Python — review with `huggingface-cli ...`
# beforehand if you're paranoid about it.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map='auto',
).eval()
print('Loaded on:', model.device)

In [ ]:
from PIL import Image
import torchvision.transforms as T
import urllib.request

# InternVL2's vision tower expects ImageNet normalisation at 448x448. The
# constants and transform below match the official chat.py demo.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
transform = T.Compose([
    T.Lambda(lambda im: im.convert('RGB')),
    T.Resize((448, 448), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

SRC = 'https://ultralytics.com/images/bus.jpg'
if SRC.startswith('http'):
    urllib.request.urlretrieve(SRC, '/tmp/aurum-internvl-input.jpg')
    SRC = '/tmp/aurum-internvl-input.jpg'
img = Image.open(SRC)
pixel_values = transform(img).unsqueeze(0).to(torch.bfloat16).to(model.device)
img

In [ ]:
# Multi-turn chat: model.chat() is InternVL's own helper that handles the
# image token insertion + history threading for us.
history = None
for question in [
    'Describe this image in one sentence.',
    'How many people are visible?',
    'What is the vehicle in the foreground?',
]:
    response, history = model.chat(
        tokenizer,
        pixel_values,
        question,
        generation_config=dict(max_new_tokens=256, do_sample=False),
        history=history,
        return_history=True,
    )
    print('Q:', question)
    print('A:', response)
    print()